## Análisis previo de los datos

Empezamos importando las librerías y módulos necesarios para hacer una regresión logística sobre el dataset [E-commerce Shipping Data](https://www.kaggle.com/datasets/prachi13/customer-analytics)

In [15]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

Creamos una carpeta en Drive llamada Datasets y subimos en ella la carpeta obtenida en Kaggle, dentro de la cual está el archivo Train.csv que es con el que vamos a trabajar. Conectamos el notebook a Drive para y ponemos la ruta del archivo en la función read_csv

In [8]:
from google.colab import drive
drive.mount('/content/drive')
import os
path = "/content/drive/MyDrive/Datasets/Shipment_arrival/"
if not os.path.exists(path):
    os.makedirs(path)
from google.colab import files
uploaded = files.upload()  # Esto abrirá una ventana para seleccionar el archivo desde tu ordenador
import shutil
shutil.move("Train.csv", "/content/drive/MyDrive/Datasets/Shipment_arrival/Train.csv")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Saving Train.csv to Train (1).csv


'/content/drive/MyDrive/Datasets/Shipment_arrival/Train.csv'

In [9]:
data = pd.read_csv('/content/drive/MyDrive/Datasets/Shipment_arrival/Train.csv')

# Si prefieres subir el dataset directamente a la carpeta temporal del notebook
# solo tienes que ejecutar esto:
#
# data = pd.read_csv('/content/Train.csv)


Echamos un vistazo inicial a la tabla de datos

In [10]:
data

,ID,Warehouse_block,Mode_of_Shipment,Customer_care_calls,Customer_rating,Cost_of_the_Product,Prior_purchases,Product_importance,Gender,Discount_offered,Weight_in_gms,Reached.on.Time_Y.N
0,1,D,Flight,4,2,177,3,low,F,44,1233,1
1,2,F,Flight,4,5,216,2,low,M,59,3088,1
2,3,A,Flight,2,2,183,4,low,M,48,3374,1
3,4,B,Flight,3,3,176,4,medium,M,10,1177,1
4,5,C,Flight,2,2,184,3,medium,F,46,2484,1
...,...,...,...,...,...,...,...,...,...,...,...,...
10994,10995,A,Ship,4,1,252,5,medium,F,1,1538,1
10995,10996,B,Ship,4,1,232,5,medium,F,6,1247,0
10996,10997,C,Ship,5,4,242,5,low,F,4,1155,0
10997,10998,F,Ship,5,2,223,6,medium,M,2,1210,0


Revisamos los tipos de valores que tenemos. hay varios valores de tipo categórico que tendremos que pasar a numérico.

In [17]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10999 entries, 0 to 10998
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Warehouse_block      10999 non-null  object
 1   Mode_of_Shipment     10999 non-null  object
 2   Customer_care_calls  10999 non-null  int64 
 3   Customer_rating      10999 non-null  int64 
 4   Cost_of_the_Product  10999 non-null  int64 
 5   Prior_purchases      10999 non-null  int64 
 6   Product_importance   10999 non-null  object
 7   Gender               10999 non-null  int64 
 8   Discount_offered     10999 non-null  int64 
 9   Weight_in_gms        10999 non-null  int64 
 10  Reached.on.Time_Y.N  10999 non-null  int64 
dtypes: int64(8), object(3)
memory usage: 945.4+ KB


Revisamos también los valores estadísticos representativos de cada variable para comprobar si es necesaria una normalización o escalado de los datos

In [16]:
data.describe()

,Customer_care_calls,Customer_rating,Cost_of_the_Product,Prior_purchases,Gender,Discount_offered,Weight_in_gms,Reached.on.Time_Y.N
count,10999.000000,10999.000000,10999.000000,10999.000000,10999.000000,10999.000000,10999.000000,10999.000000
mean,4.054459,2.990545,210.196836,3.567597,0.495863,13.373216,3634.016729,0.596691
std,1.141490,1.413603,48.063272,1.522860,0.500006,16.205527,1635.377251,0.490584
min,2.000000,1.000000,96.000000,2.000000,0.000000,1.000000,1001.000000,0.000000
25%,3.000000,2.000000,169.000000,3.000000,0.000000,4.000000,1839.500000,0.000000
50%,4.000000,3.000000,214.000000,3.000000,0.000000,7.000000,4149.000000,1.000000
75%,5.000000,4.000000,251.000000,4.000000,1.000000,10.000000,5050.000000,1.000000
max,7.000000,5.000000,310.000000,10.000000,1.000000,65.000000,7846.000000,1.000000


## Preprocesado de los datos

Eliminamos la columna del identificador de cada registro porque no nos aporta nada

In [13]:
data = data.drop('ID', axis=1)

Empezamos a convertir variables categóricas en numéricas. Para el caso del género, solo hay dos valores posibles, así que utilizamos la función replace

In [14]:
data['Gender'] = data['Gender'].replace({'F': 0, 'M': 1})

<ipython-input-14-37d7d97fd1d1>:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data['Gender'] = data['Gender'].replace({'F': 0, 'M': 1})


Para los campos que contienen más de dos categorías, recurrimos a una técnica muy útil denominada one-hot encoding, que consiste en desdoblar una columna en tantas como categorías tenga, y cada instancia tendrá valor 1 para la categoría del valor original y valor 0 en las demás.

In [18]:
def onehot_encode(df, column):
    df = df.copy()
    dummies = pd.get_dummies(df[column], prefix=column)
    df = pd.concat([df, dummies], axis=1)
    df = df.drop(column, axis=1)
    return df

for column in ['Warehouse_block', 'Mode_of_Shipment', 'Product_importance']:
    data = onehot_encode(data, column=column)

In [19]:
data

,Customer_care_calls,Customer_rating,Cost_of_the_Product,Prior_purchases,Gender,Discount_offered,Weight_in_gms,Reached.on.Time_Y.N,Warehouse_block_A,Warehouse_block_B,Warehouse_block_C,Warehouse_block_D,Warehouse_block_F,Mode_of_Shipment_Flight,Mode_of_Shipment_Road,Mode_of_Shipment_Ship,Product_importance_high,Product_importance_low,Product_importance_medium
0,4,2,177,3,0,44,1233,1,False,False,False,True,False,True,False,False,False,True,False
1,4,5,216,2,1,59,3088,1,False,False,False,False,True,True,False,False,False,True,False
2,2,2,183,4,1,48,3374,1,True,False,False,False,False,True,False,False,False,True,False
3,3,3,176,4,1,10,1177,1,False,True,False,False,False,True,False,False,False,False,True
4,2,2,184,3,0,46,2484,1,False,False,True,False,False,True,False,False,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10994,4,1,252,5,0,1,1538,1,True,False,False,False,False,False,False,True,False,False,True
10995,4,1,232,5,0,6,1247,0,False,True,False,False,False,False,False,True,False,False,True
10996,5,4,242,5,0,4,1155,0,False,False,True,False,False,False,False,True,False,True,False
10997,5,2,223,6,1,2,1210,0,False,False,False,False,True,False,False,True,False,False,True


Separamos el dataset en los datos de entrada 'X' y los de salida 'y'

In [20]:
y = data['Reached.on.Time_Y.N']
X = data.drop('Reached.on.Time_Y.N', axis=1)

Y también hacemos la separación entre los datos que usaremos para el entrenamiento 'train' y los que reservamos para evaluar después el modelo 'test'

In [21]:
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.7, shuffle=True, random_state=1)

Para varias de las técnicas que se van a aplicar, es necesario que los datos estén normalizados. En general, esto se hace operando con la media y la varianza, pero si no conoces la técnica matemática, no te preocupes, el módulo StandardScaler lo hace por ti. Puedes consultar la [documentación](https://scikit-learn.org/stable/modules/preprocessing.html) para saber más.

In [22]:
scaler = StandardScaler()
scaler.fit(X_train)
X_train = pd.DataFrame(scaler.transform(X_train), index=X_train.index, columns=X_train.columns)
X_test = pd.DataFrame(scaler.transform(X_test), index=X_test.index, columns=X_test.columns)

In [23]:
X_train

,Customer_care_calls,Customer_rating,Cost_of_the_Product,Prior_purchases,Gender,Discount_offered,Weight_in_gms,Warehouse_block_A,Warehouse_block_B,Warehouse_block_C,Warehouse_block_D,Warehouse_block_F,Mode_of_Shipment_Flight,Mode_of_Shipment_Road,Mode_of_Shipment_Ship,Product_importance_high,Product_importance_low,Product_importance_medium
4177,-0.051017,1.422394,-1.289402,-0.371349,1.001690,-0.266367,0.803593,-0.443829,-0.446551,-0.445505,-0.451563,1.411463,-0.434148,-0.441940,0.689133,-0.308738,1.039891,-0.873034
1616,-0.923855,0.715649,-1.874430,-0.371349,-0.998313,3.091967,-1.245664,2.253122,-0.446551,-0.445505,-0.451563,-0.708485,-0.434148,-0.441940,0.689133,-0.308738,1.039891,-0.873034
2775,-0.051017,-1.404585,-0.683481,-0.371349,1.001690,0.355547,-1.064775,-0.443829,2.239385,-0.445505,-0.451563,-0.708485,-0.434148,-0.441940,0.689133,-0.308738,1.039891,-0.873034
10272,-0.051017,0.715649,-1.059570,-0.371349,1.001690,-0.515133,0.489031,-0.443829,-0.446551,-0.445505,2.214530,-0.708485,2.303364,-0.441940,-1.451099,-0.308738,-0.961639,1.145431
6836,-0.051017,0.008904,0.758195,-0.371349,-0.998313,-0.763898,0.963634,2.253122,-0.446551,-0.445505,-0.451563,-0.708485,-0.434148,2.262749,-1.451099,3.238988,-0.961639,-0.873034
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7813,-0.051017,-1.404585,0.611938,-0.371349,-0.998313,-0.577324,0.701805,-0.443829,-0.446551,-0.445505,-0.451563,1.411463,2.303364,-0.441940,-1.451099,-0.308738,1.039891,-0.873034
10955,-1.796694,1.422394,-1.790855,0.290460,1.001690,-0.515133,1.363429,-0.443829,-0.446551,-0.445505,-0.451563,1.411463,2.303364,-0.441940,-1.451099,-0.308738,1.039891,-0.873034
905,-0.051017,0.715649,-1.331190,-0.371349,1.001690,-0.017602,-1.531407,-0.443829,-0.446551,-0.445505,-0.451563,1.411463,-0.434148,-0.441940,0.689133,-0.308738,1.039891,-0.873034
5192,0.821822,0.715649,0.465681,-0.371349,1.001690,-0.328559,0.879015,2.253122,-0.446551,-0.445505,-0.451563,-0.708485,-0.434148,2.262749,-1.451099,-0.308738,1.039891,-0.873034


In [33]:
X_train.describe()

,Customer_care_calls,Customer_rating,Cost_of_the_Product,Prior_purchases,Gender,Discount_offered,Weight_in_gms,Warehouse_block_A,Warehouse_block_B,Warehouse_block_C,Warehouse_block_D,Warehouse_block_F,Mode_of_Shipment_Flight,Mode_of_Shipment_Road,Mode_of_Shipment_Ship,Product_importance_high,Product_importance_low,Product_importance_medium
count,7.699000e+03,7.699000e+03,7.699000e+03,7.699000e+03,7.699000e+03,7.699000e+03,7.699000e+03,7.699000e+03,7.699000e+03,7.699000e+03,7.699000e+03,7.699000e+03,7.699000e+03,7.699000e+03,7.699000e+03,7.699000e+03,7.699000e+03,7.699000e+03
mean,-4.245352e-17,1.661225e-16,-2.713334e-16,-4.337642e-17,7.383221e-18,3.691611e-18,1.476644e-17,-3.783901e-17,6.183448e-17,-2.076531e-17,3.783901e-17,-2.768708e-17,-1.107483e-16,1.476644e-17,-1.845805e-18,3.922336e-17,-8.906010e-17,-4.752949e-17
std,1.000065e+00,1.000065e+00,1.000065e+00,1.000065e+00,1.000065e+00,1.000065e+00,1.000065e+00,1.000065e+00,1.000065e+00,1.000065e+00,1.000065e+00,1.000065e+00,1.000065e+00,1.000065e+00,1.000065e+00,1.000065e+00,1.000065e+00,1.000065e+00
min,-1.796694e+00,-1.404585e+00,-2.396776e+00,-1.033157e+00,-9.983129e-01,-7.638981e-01,-1.619705e+00,-4.438286e-01,-4.465512e-01,-4.455047e-01,-4.515632e-01,-7.084849e-01,-4.341476e-01,-4.419404e-01,-1.451099e+00,-3.087384e-01,-9.616392e-01,-8.730339e-01
25%,-9.238554e-01,-6.978405e-01,-8.506313e-01,-3.713487e-01,-9.983129e-01,-5.773240e-01,-1.094821e+00,-4.438286e-01,-4.465512e-01,-4.455047e-01,-4.515632e-01,-7.084849e-01,-4.341476e-01,-4.419404e-01,-1.451099e+00,-3.087384e-01,-9.616392e-01,-8.730339e-01
50%,-5.101668e-02,8.904305e-03,1.104858e-01,-3.713487e-01,-9.983129e-01,-3.907499e-01,3.105945e-01,-4.438286e-01,-4.465512e-01,-4.455047e-01,-4.515632e-01,-7.084849e-01,-4.341476e-01,-4.419404e-01,6.891329e-01,-3.087384e-01,-9.616392e-01,-8.730339e-01
75%,8.218220e-01,7.156491e-01,8.626645e-01,2.904600e-01,1.001690e+00,-2.041758e-01,8.566338e-01,-4.438286e-01,-4.465512e-01,-4.455047e-01,-4.515632e-01,1.411463e+00,-4.341476e-01,-4.419404e-01,6.891329e-01,-3.087384e-01,1.039891e+00,1.145431e+00
max,2.567499e+00,1.422394e+00,2.074508e+00,4.261312e+00,1.001690e+00,3.216350e+00,2.577531e+00,2.253122e+00,2.239385e+00,2.244645e+00,2.214530e+00,1.411463e+00,2.303364e+00,2.262749e+00,6.891329e-01,3.238988e+00,1.039891e+00,1.145431e+00


In [25]:
y_train

,Reached.on.Time_Y.N
4177,1
1616,1
2775,1
10272,0
6836,0
...,...
7813,0
10955,0
905,1
5192,1


## Generación del modelo y entrenamiento

Creamos un objeto model de la clase LogisticRegression. Con ésto, estamos generando una expresión matemática con coeficientes sin fijar, que después del entrenamiento con la función "fit" ya sí tendrán valores fijos

In [26]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()

In [27]:
model.fit(X_train,y_train)

LogisticRegression()

Pedimos que se nos muestre la tasa de acierto con los datos de test

In [28]:
print('Model score: ', model.score(X_test,y_test))

Model score:  0.6348484848484849


Como no es muy buena métrica, ahora vamos a comparar con otras técnicas de aprendizaje automático

## Comparativa

Incluimos también el modelo XGBClassifier, de la librería xgboost, porque es un modelo de clasificación que tiene muy buen desempeño en este tipo de problemas. Si quieres saber más sobre la técnica del Gradient Boosting, puedes leer sobre ello [aquí](https://www.datacamp.com/tutorial/xgboost-in-python).

In [29]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import LinearSVC, SVC
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

/usr/local/lib/python3.11/dist-packages/dask/dataframe/__init__.py:42: FutureWarning: 
Dask dataframe query planning is disabled because dask-expr is not installed.

You can install it with `pip install dask[dataframe]` or `conda install dask`.
This will raise in a future version.

  warnings.warn(msg, FutureWarning)


Creamos los modelos con un diccionario y ejecutamos el entrenamiento de cada uno a través de un bucle for

In [30]:
models = {
    "                   Logistic Regression": LogisticRegression(),
    "                   K-Nearest Neighbors": KNeighborsClassifier(),
    "                         Decision Tree": DecisionTreeClassifier(),
    "Support Vector Machine (Linear Kernel)": LinearSVC(),
    "   Support Vector Machine (RBF Kernel)": SVC(),
    "                        Neural Network": MLPClassifier(),
    "                         Random Forest": RandomForestClassifier(),
    "                     Gradient Boosting": GradientBoostingClassifier(),
    "                               XGBoost": XGBClassifier(eval_metric='mlogloss'),
    "                              LightGBM": LGBMClassifier()
}

for name, model in models.items():
    model.fit(X_train, y_train)
    print(name + " trained.")

                   Logistic Regression trained.
                   K-Nearest Neighbors trained.
                         Decision Tree trained.
Support Vector Machine (Linear Kernel) trained.
   Support Vector Machine (RBF Kernel) trained.


/usr/local/lib/python3.11/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


                        Neural Network trained.
                         Random Forest trained.
                     Gradient Boosting trained.
                               XGBoost trained.
[LightGBM] [Info] Number of positive: 4605, number of negative: 3094
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001806 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 592
[LightGBM] [Info] Number of data points in the train set: 7699, number of used features: 18
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.598130 -> initscore=0.397678
[LightGBM] [Info] Start training from score 0.397678
                              LightGBM trained.


Presentación de las métricas

In [31]:
for name, model in models.items():
    print(name + ": {:.2f}%".format(model.score(X_test, y_test) * 100))

                   Logistic Regression: 63.48%
                   K-Nearest Neighbors: 63.67%
                         Decision Tree: 64.73%
Support Vector Machine (Linear Kernel): 63.70%
   Support Vector Machine (RBF Kernel): 65.12%
                        Neural Network: 63.55%
                         Random Forest: 64.70%
                     Gradient Boosting: 68.03%
                               XGBoost: 65.30%
                              LightGBM: 66.70%


Podemos ver que la técnica de EXtreme Gradient Boosting o XGBoost es la que nos da mejor tasa de acierto, por encima de Random Forest, que suele ser la más valorada dentro de Scikit Learn.

Ejemplo adaptado a partir del notebook en Kaggle.com: https://www.kaggle.com/code/gcdatkin/shipment-arrival-prediction/notebook